[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geo-di-lab/emerge-geoai-es/blob/main/docs/ch6/lesson2.ipynb)

# Mapear tus datos

Este archivo de plantilla muestra cómo crear mapas estáticos e interactivos.

In [ ]:
import geopandas as gpd
import folium
import matplotlib.pyplot as plt

# Instalar contextily en Google Colab
!pip install contextily

import contextily as cx

## Cargar tus datos

En Google Colab, haz clic en el ícono de carpeta ubicado a la izquierda de la pantalla. Si aparece el mensaje `Connecting to a runtime to enable file browsing`, espera algunos segundos mientras Google Colab se conecta y carga.

Haz clic en el ícono **Upload**, que es el primero de izquierda a derecha, y selecciona el archivo en tu computadora.

Aparecerá el mensaje `Warning: Ensure that your files are saved elsewhere. This runtime's files will be deleted when this runtime is terminated.` Esto ocurre porque Google Colab crea una copia temporal de los datos en tu navegador. Esa copia se eliminará cuando cierres el navegador o finalice la sesión. El archivo original de tu computadora no cambiará, aunque modifiques los datos dentro de Google Colab.

A continuación se muestran ejemplos para cargar distintos tipos de datos espaciales.

1. Shapefile (`.shp`):

Si subiste un shapefile comprimido:

```python
data = gpd.read_file("nombre_de_tu_archivo.zip")
```

Si subiste por separado los archivos del shapefile (`.SHP`, `.DBF`, `.SHX`, entre otros):

```python
data = gpd.read_file("nombre_de_tu_archivo.shp")
```

2. GeoJSON (`.geojson`):

```python
data = gpd.read_file("nombre_de_tu_archivo.geojson")
```

3. GeoPackage (`.gpkg`):

```python
data = gpd.read_file(
    "nombre_de_tu_archivo.gpkg",
    layer="nombre_de_tu_capa"
)
```

4. Geodatabase (`.gdb`):

Primero debes comprimir la geodatabase en un archivo ZIP.

```python
data = gpd.read_file(
    "nombre_de_tu_geodatabase_comprimida.zip",
    layer="nombre_de_tu_capa"
)
```

In [ ]:
# Escribe aquí el código para cargar tus datos
data = None

# Por ejemplo:
# data = gpd.read_file("nombre_de_tu_archivo.geojson")

En este ejemplo utilizaremos el conjunto de datos de GLOBE Observer Mosquito Habitat Mapper, filtrado para incluir solamente las observaciones de Florida:

```python
data = gpd.read_file(
    "https://github.com/geo-di-lab/emerge-lessons/raw/"
    "refs/heads/main/docs/data/globe_mosquito.zip"
)

fl = gpd.read_file(
    "https://github.com/geo-di-lab/emerge-lessons/raw/"
    "refs/heads/main/docs/data/florida_boundary.geojson"
)[["geometry"]]

data = (
    gpd.sjoin(
        data,
        fl,
        how="inner",
        predicate="intersects"
    )
    .drop(columns=["index_right"])
    .reset_index(drop=True)
)
```

In [ ]:
# Mostrar las primeras cinco filas de la tabla
if data is None:
    raise ValueError(
        "Primero carga tus datos en la celda anterior."
    )

data.head()

In [ ]:
# Mostrar estadísticas como el promedio, mínimo y máximo
# de cada columna numérica
if data is None:
    raise ValueError(
        "Primero carga tus datos en la celda anterior."
    )

data.describe()

Crea un mapa sencillo para observar los datos.

In [ ]:
if data is None:
    raise ValueError(
        "Primero carga tus datos."
    )

data.plot()
plt.show()

Agrega un mapa base mediante [contextily](https://contextily.readthedocs.io/en/latest/).

Utilizamos `.to_crs(epsg=3857)` para transformar los datos a un sistema de referencia de coordenadas compatible con el mapa base que aparecerá en el fondo.

In [ ]:
if data is None:
    raise ValueError(
        "Primero carga tus datos."
    )
if data.crs is None:
    raise ValueError(
        "Los datos no tienen un sistema de referencia de coordenadas definido."
    )

ax = data.to_crs(
    epsg=3857
).plot(
    color="gold",  # Nombre del color de los puntos
    edgecolor="black",
    markersize=25
)

cx.add_basemap(ax)
ax.axis("off")
plt.show()

Puedes personalizar el color de los puntos. Esta imagen muestra una lista de colores con nombre disponibles en Matplotlib:

![Lista de colores](https://matplotlib.org/stable/_images/sphx_glr_named_colors_003_2_00x.png)

## Crear un mapa interactivo con [Folium](https://python-visualization.github.io/folium/latest/)

### Si tu conjunto de datos contiene puntos

En este ejemplo continuaremos utilizando el conjunto de datos de GLOBE Observer Mosquito Habitat Mapper correspondiente a Florida.

En el siguiente código puedes definir `popup_content`, que es la información que aparecerá cuando hagas clic en un punto. Por ejemplo:

```python
popup_content = f"""<b>Etiqueta 1:</b> {row["Column1"]}<br>
<b>Etiqueta 2:</b> {row["Column2"]}<br>
<b>Etiqueta 3:</b> {row["Column3"]}"""
```

En este código, `<b>Nombre:</b>` hace que la palabra `Nombre:` aparezca en negrita porque está ubicada entre `<b>` y `</b>`. La etiqueta `<br>` crea un salto de línea y coloca el siguiente texto en una línea nueva.

Para personalizar el código, reemplaza `Column1`, `Column2` y `Column3` por los nombres exactos de las columnas de tu conjunto de datos. Después, reemplaza `Etiqueta 1`, `Etiqueta 2` y `Etiqueta 3` por los textos que deseas mostrar para describir cada columna.

Las etiquetas pueden ser iguales a los nombres de las columnas o pueden ser más descriptivas. Por ejemplo, si una columna se llama `lat`, puedes utilizar `Latitud` como etiqueta.

In [ ]:
if data is None:
    raise ValueError(
        "Primero carga tus datos."
    )

required_columns = [
    "Column1",
    "Column2",
    "Column3"
]

missing_columns = [
    column
    for column in required_columns
    if column not in data.columns
]

if missing_columns:
    raise ValueError(
        "Reemplaza Column1, Column2 y Column3 por nombres "
        "reales de columnas antes de ejecutar esta celda."
    )

if not data.geometry.geom_type.isin(["Point"]).all():
    raise ValueError(
        "Este ejemplo requiere un conjunto de datos formado por puntos."
    )

# Crear un mapa vacío centrado en Florida
map = folium.Map(
    location=[28.263363, -83.497652],
    tiles="CartoDB positron",
    zoom_start=7
)

# Agregar cada punto como marcador
for _, row in data.iterrows():
    popup_content = f"""<b>Etiqueta 1:</b> {row["Column1"]}<br>
<b>Etiqueta 2:</b> {row["Column2"]}<br>
<b>Etiqueta 3:</b> {row["Column3"]}"""

    folium.Marker(
        location=[
            row.geometry.y,
            row.geometry.x
        ],
        popup=folium.Popup(
            popup_content,
            max_width=300
        )
    ).add_to(map)

display(map)

### Si tu conjunto de datos contiene polígonos

En este ejemplo utilizaremos los límites de los condados de Florida:

```python
data = gpd.read_file(
    "https://github.com/geo-di-lab/emerge-lessons/raw/"
    "refs/heads/main/docs/data/florida_counties.geojson"
)
```

In [ ]:
# Escribe el nombre de una columna que deseas mostrar
# en la ventana emergente
column_name = ""

In [ ]:
if data is None:
    raise ValueError(
        "Primero carga tus datos."
    )
if not column_name:
    raise ValueError(
        "Escribe el nombre de una columna en column_name."
    )
if column_name not in data.columns:
    raise ValueError(
        f"No se encontró la columna '{column_name}'."
    )
if data.crs is None:
    raise ValueError(
        "Los datos no tienen un sistema de referencia de coordenadas definido."
    )
if not data.geometry.geom_type.isin(
    ["Polygon", "MultiPolygon"]
).all():
    raise ValueError(
        "Este ejemplo requiere un conjunto de datos formado por polígonos."
    )

# Crear un mapa vacío centrado en Florida
map = folium.Map(
    location=[28.263363, -83.497652],
    tiles="CartoDB positron",
    zoom_start=7
)

# Agregar cada polígono al mapa
for _, row in data.iterrows():
    # Simplificar la geometría para que el mapa cargue más rápido
    simplified_geometry = gpd.GeoSeries(
        [row["geometry"]],
        crs=data.crs
    ).simplify(
        tolerance=0.001
    )

    geo_json = folium.GeoJson(
        data=simplified_geometry.to_json(),
        style_function=lambda feature: {
            "fillColor": "blue"
        }
    )

    folium.Popup(
        str(row[column_name])
    ).add_to(geo_json)

    geo_json.add_to(map)

display(map)

## Referencias

- [Introducción a GeoPandas](https://geopandas.org/en/stable/getting_started/introduction.html)
- [Creación de mapas de polígonos con Folium](https://geopandas.org/en/stable/gallery/polygon_plotting_with_folium.html)
- [Guía de introducción a contextily](https://contextily.readthedocs.io/en/latest/intro_guide.html)